# run_experiment Module

A separate module is use to run the Bayesian Optimisation within each problem folder. First, relevant modules are imported, including relevant classes from the base, optimiser, neural_optimiser and plotter modules.

When the experiment is run, the raw data are loaded, then the experimental configuration is set up. This allows for a choice between a Gaussian Process and a Neural Network surrogate, as well as custom hyperparameters. Then, the suggest function of the optimiser class is run, printing the best model and suggested next point. Finally, the plotter module is called to plot all kernels in 1-D slices, with suggested outputs from each using both UCB and EI.



## Initial Setup

The first section imports relevant modules and establishes the current directory

In [ ]:
import sys
import numpy as np
from pathlib import Path
from dataclasses import dataclass, field
from typing import Literal

current_dir = Path(__file__).resolve().parent
sys.path.append(str(current_dir))

try:
    from core.base import UCB, ExpectedImprovement
    from core.optimiser import BayesianOptimizer, lbfgs_optimizer
    from core.neural_optimiser import NeuralBayesianOptimizer, de_optimizer
    from core.plotter import Plotter
except ImportError as e:
    print("Error: Could not import 'core' library. Ensure file structure is correct.")
    raise e

## Experiment Configuration

The ExperimentConfig class is created. This class contains the choice of surrogate (GP or NN), as well as hyperparameters for the NN surrogate if applicable. The experiment config also defines the data directory, and history directory, which contains all previously guessed points, as well as basic details about bounds, transforms and random seed.

The load_data function simply opens the inputs and outputs files from the data directory

In [ ]:
@dataclass
class ExperimentConfig:

    model_type: Literal['GP', 'NN']
    nn_max_trials: int
    nn_patience: int
    nn_ensemble_size: int

    data_dir: Path = field(default=current_dir / "data")
    history_file: Path = field(default=current_dir / "history.csv")
    bounds: tuple[float, float] = (0.0, 1.0)
    symlog_transform_y: bool = False
    log_transform_y: bool = False
    seed: int = 42

    def load_data(self) -> tuple[np.ndarray, np.ndarray]:
        input_file = self.data_dir / "inputs.npy"
        output_file = self.data_dir / "outputs.npy"
        if not input_file.exists(): 
            raise FileNotFoundError(f"Data inputs.npy not found in {self.data_dir}")
        return np.load(input_file), np.load(output_file)

## Factory Functions

These functions initialise the BayesianOptimizer or NeuralBayesianOptimizer classes from the optimiser/neural_optimizer modules with the current ExperimentConfig and runs the evaluate_models/build_ensemble functions from each class.

In [ ]:
def setup_gp(cfg: ExperimentConfig, X, y):
    print("\n--- Initializing GP Optimizer ---")
    opt = BayesianOptimizer(X, y, cfg.bounds, cfg.symlog_transform_y, cfg.log_transform_y, cfg.seed, cfg.history_file)
    models, df_ranks = opt.evaluate_models()
    print(df_ranks[["name", "mean_score"]].head(3))
    return opt, models, df_ranks, lbfgs_optimizer

def setup_nn(cfg: ExperimentConfig, X, y):
    print("\n--- Initializing NN Optimizer ---")
    opt = NeuralBayesianOptimizer(X, y, cfg.bounds, cfg.symlog_transform_y, cfg.log_transform_y, cfg.seed, cfg.history_file)
    ensemble = opt.build_ensemble(cfg.nn_max_trials, cfg.nn_ensemble_size, cfg.nn_patience)
    return opt, ensemble, None, de_optimizer

## Run Function

The run function loads the raw data from the ExperimentConfig class and checks the dimensions are compatible and runs the setup function. In the acquisitions setup, the user can modify the κ and ξ hyperparameters for UCB and EI. The structure of the code is set up to allow for additional acquisition functions to be seemlessly included.

Then for each acquisition function, the best model name and the suggested next point are printed. The plotter module is called to produce 1-D slices for each and show the suggested next points for each kernel.

In [ ]:
def run(cfg: ExperimentConfig):
    
    try:
        X, y = cfg.load_data()
  
        print(f"[{cfg.model_type}] Data: {X.shape}, {y.shape} Max Y: {np.max(y):.4f}")
    except Exception as e:
        print(f"Data Error: {e}")
        return

    if cfg.model_type == 'GP':
        opt, models, df_ranks, strategy = setup_gp(cfg, X, y)
    elif cfg.model_type == 'NN':
        opt, models, df_ranks, strategy = setup_nn(cfg, X, y)
    else:
        raise ValueError(f"Unknown Type: {cfg.model_type}")

    acquisitions = [UCB(kappa=1.96), ExpectedImprovement(xi=0.05)]
    
    for acq in acquisitions:
        print(f"\n--- Strategy: {acq.name} ---")
        
        if cfg.model_type == 'GP':
            best_name = df_ranks.iloc[0]["name"]
            active_model = models[best_name]
        else:
            active_model = models 

        res = opt.suggest(active_model, acq, strategy)
        print(f"   Next: {np.round(res.next_coords, 4)} | Score: {res.score:.4f}")

        Plotter.plot_model_comparison(
            optimizer=opt,
            models_dict=models,
            df_ranks=df_ranks,
            acquisition=acq,
            optimizer_strategy=strategy
        )

    print("\nRun Complete.")

## Main Execution

The main execution of the module allows the user to select the desired BO type and set up NN hyperparameters if required. The ExperimentConfig class is initialised and the experiment is run

In [ ]:
if __name__ == "__main__":
    
    SELECTED_MODEL = 'GP'

    NN_TRIALS   = 50
    NN_ENSEMBLE = 5 
    NN_PATIENCE = 15 
    
    SEED = 42
    
    config = ExperimentConfig(
        model_type=SELECTED_MODEL,
        nn_max_trials=NN_TRIALS,
        nn_ensemble_size=NN_ENSEMBLE,
        nn_patience=NN_PATIENCE,
        seed=SEED
    )
    
    run(config)